# Using State Infromation to Assess Reliability of Complex Systems:

Suppose there exists a network with the following structure:

In [ ]:
from matplotlib import pyplot as plt
import networkx as nx
G = nx.DiGraph()
G.add_edges_from([('start', 1), (1,3), (3, 'end'), ('start', 2), (2,4), (4,'end')])
pos = {'start': (0, 1), 1:(1,2), 2:(1,0), 3:(2,2), 4:(2,0), 'end':(3,1)}
style = {'pos': pos, 'node_color':'skyblue', 'node_size':1200, 'with_labels':True, 'font_weight':'bold'}
plt.figure(figsize=(10, 4))
nx.draw(G, **style)

## Current Analysis:
Here is the current process:

1. Collect statistics on each node seperately to get activity percentages for a given time interval.
2. Evaluate each node to get the percentage of active time $A_i = \lambda_i / (\lambda_i+\mu_i): \lambda:$ total up time, $\lambda+\mu$: total time
3. Evaluate each distinct chain for a total measurement via $C_1 = A_1\cdot A_3$ and $C_2 = A_2\cdot A_4$
4. Evaluate the aggregrate performance of all chains with $1 - (1-C_1)(1-C_2)$

## Issues with this Approach:

1. **Data Collection:** The whole state of the system is not contextualized. If the analysis is conducted to assess the whole system, then measurements need to be taken to reflect the combined status of all nodes within that system. A failure to do so results in the inability to capture joint information. Without this information, the node performance cannot be contextualized within the whole system. This results in analysts being forced to make unideal assumptions about the composition of performance or use alternative methods to assess context. 

2. **A non-generalizable chain formula:** $A_1\cdot A_3$, in terms of probability is really $P(A_1\text{ is active}) \cap P(A_3 \text{ is active})$. This is missing a normalization/contextualization/dependency term, and should be $P(A_1\text{is active})\cdot P(A_2 \text{ is active}|A_1 \text{ is active})$

2. **A non-generalizable aggregation formula:** $1 - (1-C_1)(1-C_2)$ suffers from similar problems to (2). The textbook definition of this probability is $P(C_1)+P(C_2)-P(C_1 \cap C_2)$. If the compliment rule wants to be leveraged, then either the probability $(1-C_2)$ needs to be conditioned on. The easier path is to leverage joint distributions instead. 

### Counterexample:
Suppose there exists activity with the following data:

In [ ]:
import numpy as np
data = np.zeros((4,4))
d = np.array([1,0,1,0])
one = np.ones((1,4))
for i in range(4):
    data[i, :] = d
    d = np.logical_xor(d, one)

print(f'Data:\n {data}')
# Active/Total Time
probabilities = data.sum(axis=1)/data.shape[1]
print(f'Activity:\n {probabilities}')

Based on data, it can be seen that $C_1$ was active for time $t=1, 3$ and $C_2$ was active for $t=2, 4$; *therefore, there was never a point in time in which the network was not active.* The existing process cannot account for this. *Why?* Context is ignored $\iff$ _Data_ is assumed to be independent $\iff$ No joint information is displayed

In [ ]:
probabilities

In [ ]:
# Assessment based on the first formulas:
A1, A2, A3, A4 = probabilities
C1 = A1*A3
C2 = A2*A4
print(f"C Values: {C1:.2f}, {C2:.2f}")
perf = 1 - (1-C1)*(1-C2)
print(f"Formula 1 Overall Assessment: {perf:.2f}\n")

# State based on observation:
C1_obs = np.logical_and(data[0, :], data[2, :]).astype(int)
C2_obs = np.logical_and(data[1, :], data[3, :]).astype(int)
print(f'C1 observed: {C1_obs}')
print(f'C2 observed: {C2_obs}')
total = np.logical_or(C1_obs, C2_obs).astype(int)
print(f"Network Observed: {total}")
print(f"Observed Overall Assessment: {total.sum()/len(total)}")

A formula cannot be correct if there exists a counter-example to the problem. Now, the same analysis will be conducted using the corrected formulas.

$P(A_1 \cap A_3) = P(A_1)P(A_3|A_1)$

$P(A_3|A_1) = P(A_1 \cup A_3)$

In [ ]:
# Using conditional/dependent information from the data itself
pa1 = data[0, :].sum()/data[0, :].size
cond_a1_a3 = (data[0, :] == 1)&(data[2, :] == 1)
cond_a2_a4 = (data[1, :] == 1)&(data[3, :] == 1)
cond_c1_c2 = cond_a1_a3 & cond_a2_a4
p_a1_and_a3 = np.sum(cond_a1_a3)/data.shape[1]
p_a2_and_a4 = np.sum(cond_a2_a4)/data.shape[1]
p_c1_and_c2 = np.sum(cond_c1_c2)/data.shape[1]
pa3_given_a1 = p_a1_and_a3/pa1
print("P(A3|A1) = P(A1 and A3)/P(A1)")
print(f"P(A1): {pa1}")
print(f"P(A1 and A3): {p_a1_and_a3}")
print(f"P(A3|A1): {pa1}/{p_a1_and_a3} = {pa3_given_a1:<30}\n")
print(f"C1 Assessment: {p_a1_and_a3}")
print(f"C2 Assessment: {p_a1_and_a3} (by the same logic)")

# Getting full network performance
p_c1 = p_a1_and_a3
p_c2 = p_a2_and_a4
final = p_c1 + p_c2 - p_c1_and_c2
print(f"Final Assessment: {final:.1f}")
print(f"Observed Overall Assessment: {total.sum()/len(total)}")


### Remarks: 
The revised method is able to pass the counter-example that the original formulas could not, but can it generalize to a set of example? **Yes**

If data is truly independent, i.e. observations are uncorrelated, then the functions collaspe to the previous version: 

1. $P(A_1)P(A_3|A_1) \Rightarrow P(A_1)P(A_3)$
2. $P((C_1 \cup C_2)^c)^c = P((C_1^c \cap C_2^c))^c = (P(C_1^c)P(C_3^c|C_1^c))^c \Rightarrow (P(C_1^c)P(C_3^c))^c$
2. $P(C_1 \cup C_2) = P(C_1) + P(C_2) - P(C_1 \cap C_2)$. Indep $\Rightarrow (C_1\cap C_2) = \{\emptyset\} $

### There is an issue: Data Structure

What the counter-example really exposes is the insufficently of the underlying data structure. At a small scale, a sound approach is to look at joint-pdfs. For four nodes, there are $2^4 \rightarrow 16$ possible states:

In [ ]:
# Generalized Assessment
joint_dist = np.random.rand(2, 2, 2, 2)
joint_dist /= np.sum(joint_dist) # normalize to a joint-pdf
print("Shape of 4-variable joint distribution:", joint_dist.shape)
print("Sum of all joint probabilities:", np.sum(joint_dist))
# P(A1=1, A2=0, A3=1, A4=0)
p_1010 = joint_dist[1, 0, 1, 0]
print(f"P(A1=1, A2=0, A3=1, A4=0) = {p_1010:.6f}\n")
print(f'Joint-PDF:\n {joint_dist}\n')
p1and3 = np.sum(joint_dist[1,:,1,:])
p2and4 = np.sum(joint_dist[:,1,:,1])
p_all = joint_dist[1,1,1,1]
p_pass = p1and3 + p2and4 - p_all
print(f"Overall Assessment: {p_pass}")

# Old Assessment:
ma1 = np.sum(joint_dist[1, :, :, :])
ma2 = np.sum(joint_dist[:, 1, :, :])
ma3 = np.sum(joint_dist[:, :, 1, :])
ma4 = np.sum(joint_dist[:, :, :, 1])
result = 1-(1-(ma1*ma3))*(1-(ma2*ma4))
print(f"Old Overall Assessment: {result}")


## Other Notes Regarding the Counter-example:

In the old assessment, all three cases receive the same rating:

1. When (A1, A3), (A2, A4) never have active pairs 
2. When (A1, A3), (A2, A4) always have active pairs
3. When pairs are 50/50



In [ ]:
data1, data2, data3 = np.zeros((4,4)), np.zeros((4,4)), np.ones((4,4))*0.5
d1, d2, d3 = np.array([1,0,1,0]), np.array([1,0,1,0]), np.array([1,0,1,0])
one = np.ones((1,4))
for i in range(4):
    data1[i, :] = d1
    data2[i, :] = d2
    d1 = np.logical_xor(d1, one)
    if i == 1:
        d2 = np.logical_xor(d2, one)

print(f'Data:\n {data1}')
print(f'Data:\n {data2}')
print(f'Data:\n {data3}')
# Active/Total Time
proba1 = data1.sum(axis=1)/data1.shape[1]
proba2 = data2.sum(axis=1)/data2.shape[1]
proba3 = data3.sum(axis=1)/data3.shape[1]
print(f'Activity:\n{proba1}\n{proba2}\n{proba3}')


# Why Joint PDFs are needed(?):

For the example where all $A_i = 0.5$, when are nodes up or down? This information is critical for determining what the joint-pdf is. They're needed to do analysis if you want to work with concrete probability values directly. Analysis doesn't require that, and often times, it gets in the way of analysis. 

## Is there a more intuitive way to work with this data?
**Yes, Time series analysis!** This eliminates the need to use probabilties directly. Doing so as several issues. 

1. It is not scalable to larger contexts
2. Conditional Independencies that are well known by SMEs can be exploited systematically
3. Natural Causal and Mechanistic Interpretations 

# What does ideal data look like?

In [ ]:
import pandas as pd
from matplotlib.lines import Line2D

class ContinuousTimeMarkov:
    def __init__(self, probs: np.ndarray, total_rate: float = 2.0, max_time: float = 50.0, initial_state: int = 1):
        self.probs = probs
        self.total_rate = total_rate  # Controls overall frequency of switching
        self.max_time = max_time
        self.initial_state = initial_state

    def simulate(self) -> list:
        k = len(self.probs)
        # Derive rates q01 and q10 from steady-state info
        q01 = self.probs * self.total_rate
        q10 = (1 - self.probs) * self.total_rate
        simulated_states = []
        for i in range(k):
            t = 0.0
            state = self.initial_state
            path = [(t, state)]
            while t < self.max_time:
                # Rate of leaving current state
                rate = q01[i] if state == 0 else q10[i]
                # Sample holding time from exponential distribution
                dt = np.random.exponential(1.0 / rate)
                t += dt
                if t >= self.max_time:
                    # Record final point up to max_time
                    path.append((self.max_time, state))
                    break
                # Transition to the other state
                state = 1 - state
                path.append((t, state))
            simulated_states.append(path)  
        return simulated_states

    def plot(self, states:list, title='Continuous-Time Markov Chain Simulation'):
        # Visualization for Continuous-Time Intervals
        fig, axes = plt.subplots(nrows=4, ncols=1, figsize=(10, 8), sharex=True)
        for i, (ax, path) in enumerate(zip(axes, states)):
            # Draw horizontal blocks for each continuous state segment
            for j in range(len(path) - 1):
                t_start, state = path[j]
                t_end, _ = path[j+1]
                color = '#1f77b4' if state == 1 else '#b32a08'
                ax.hlines(y=state, xmin=t_start, xmax=t_end, linewidth=6, color=color)
            ax.set_ylabel(f'Node {i+1}\n($\\pi_1$={self.probs[i]:.2f})', rotation=0, labelpad=20, ha='center', va='center', fontsize=10)
            ax.set_yticks([0, 1])
            ax.set_ylim(-0.2, 1.2)
            ax.grid(True, linestyle='--', alpha=0.4, axis='x')
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
        axes[-1].set_xlabel('Time ($t$)', fontsize=11, labelpad=10)
        # Main Title, Legend
        fig.suptitle(title, fontsize=14, fontweight='bold', y=0.96)
        legend_elements = [Line2D([0], [0], color='#1f77b4', lw=6, label='State 1'), Line2D([0], [0], color='#b32a08', lw=6, label='State 0')]
        fig.legend(handles=legend_elements, loc='upper center', bbox_to_anchor=(0.5, 0.91), ncol=2, frameon=False)
        plt.subplots_adjust(hspace=0.4, top=0.82)
        plt.show()

    @ staticmethod
    def event_list_to_df(states:list, data=[]) -> pd.DataFrame:
        # Store each node's exact event sequence as records
        for i, path in enumerate(states):
            for t, state in path:
                data.append({'node': f'node_{i+1}', 'time': t, 'state': state,})
        return pd.DataFrame(data).sort_values(by='time').reset_index(drop=True)

    def event_list_to_timegrid(self, states: list) -> pd.DataFrame:
        all_times = {0.0, self.max_time}
        for state in states:
            for t, s in state:
                all_times.add(t)
        timeline = sorted(list(all_times))
        self.timeline = timeline
        states_matrix = []
        for state in states:
            times = [s[0] for s in state]
            state_values = [s[1] for s in state]
            idx = np.searchsorted(times, timeline, side='right') - 1
            idx = np.clip(idx, 0, len(state_values) - 1)
            states_matrix.append([state_values[i] for i in idx])   
        # Transpose so rows correspond to timeline and columns correspond to nodes
        df = pd.DataFrame(states_matrix).T
        # Name columns after your nodes
        df.columns = [f"node_{i+1}" for i in range(len(states))]
        # Insert the timeline as the first column
        df.insert(0, 'timestamp', timeline)
        return df

    def calculate_collective_activity(self, df_time, condition_func=None):
        node_cols = [col for col in df_time.columns if col != 'timestamp']
        # Calculate the duration for each row (time until the next event timestamp)
        timestamps = df_time['timestamp'].values
        next_timestamps = np.append(timestamps[1:], self.max_time)
        durations = next_timestamps - timestamps
        # Determine inactive rows using either a custom condition or the default 
        if condition_func is not None:
            is_inactive = condition_func(df_time)
        else:
            # Default logic: inactive if all nodes are 0
            is_inactive = (df_time[node_cols] == 0).all(axis=1)   
        is_active = ~is_inactive                            
        # Sum total time spent in each state
        total_duration = self.max_time - timestamps[0]
        inactive_time = np.sum(durations[is_inactive])
        active_time = np.sum(durations[is_active])
        # Compute proportions
        prop_inactive = inactive_time / total_duration
        prop_active = active_time / total_duration
        return prop_inactive, prop_active

class TwoStateDiscreteMarkov:
    def __init__(self, params: np.ndarray = None, probs: np.ndarray = None, n_steps: int = 1000, initial_state: int = 1):
        self.params = params
        self.probs = probs if probs is not None else params[0]/(params[0]+params[1])
        self.n_steps = n_steps
        self.initial_state = initial_state

    def two_state_markov(self) -> pd.DataFrame:
        k = len(self.probs) if self.probs is not None else self.params.shape[1]
        states = np.zeros((k, self.n_steps), dtype=int)
        states[:, 0] = self.initial_state
        for t in range(1, self.n_steps):
            prev = states[:, t-1]
            r = np.random.rand(k)
            # Use probs if available, or fall back/adapt to params
            p = self.probs if self.probs is not None else self.params[0]
            # Transition 0 to 1 with prob p; 1 to 0 with prob (1-p)
            p_transition = np.where(prev == 0, p, 1 - p)
            states[:, t] = np.where(r < p_transition, 1 - prev, prev)
        return pd.DataFrame(states)

    def plot(self, df):
        fig, axes = plt.subplots(nrows=4, ncols=1, figsize=(11, 8), sharex=True)
        for i, ax in enumerate(axes):
            row = df.iloc[i].values
            change_indices = np.where(np.diff(row) != 0)[0] + 1
            segments = np.split(row, change_indices)
            idx_splits = np.split(np.arange(len(row)), change_indices)
            # Draw horizontal bars for active segments
            for seg, idxs in zip(segments, idx_splits):
                if len(idxs) > 0:
                    if seg[0] == 1:
                        ax.hlines(y=1, xmin=idxs[0], xmax=idxs[-1], linewidth=6, color='#1f77b4')
                    elif seg[0] == 0:
                        ax.hlines(y=0, xmin=idxs[0], xmax=idxs[-1], linewidth=6, color="#b32a08")

            ax.set_ylabel(f'Node {i+1}\n($p$={self.probs[i]:.2f})', rotation=0, labelpad=25, ha='right', va='center', fontsize=10)
            ax.set_yticks([0, 1])
            ax.set_ylim(-0.2, 1.2)
            ax.grid(True, linestyle='--', alpha=0.4, axis='x')
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
        axes[-1].set_xlabel('Time Steps', fontsize=11, labelpad=10)
        # Main Title
        fig.suptitle('Discrete Time Markov States', fontsize=14, fontweight='bold', y=0.96)
        # Global Figure Legend
        legend_elements = [Line2D([0], [0], color='#1f77b4', lw=6, label='State 1'), Line2D([0], [0], color='#b32a08', lw=6, label='State 0')]
        fig.legend(handles=legend_elements, loc='upper center', bbox_to_anchor=(0.5, 0.91), ncol=2, frameon=False)
        plt.subplots_adjust(hspace=0.4, top=0.82)
        plt.show()


In [ ]:
# Generate parameters and calculate steady-state probabilities
np.random.seed(123)
params = np.array([np.random.normal(40, 5, size=4), np.random.normal(20, 3, size=4)])
probs = params[0] / (params[0] + params[1])  # Target steady-state probs for State 1
# Run simulation
ctmc = ContinuousTimeMarkov(probs=probs, max_time=100.0)
states = ctmc.simulate()
print(f'Example state data: {states[0][:5]}')
df = ctmc.event_list_to_df(states)
ctmc.plot(states, title='Example Node Activity Timelines')

In [ ]:
print(df.head(10))

## Anwsering the First Question:
Data in this format means that bayesian analysis, time series analysis, markov models, and others can drive insight. These types of analysis lead to complex insights, but also can anwser questions that seemed difficult before in an easy way. For example, in this small example, get an assessment of network performance, all that has to be done is to look at the timeline, see how often the active vs inactive conditions were met, and then look at the fraction of time in which those states happened.

In [ ]:
# New Condition: (1 OR 3) AND (2 OR 4) inactive = (1 AND 3) OR (2 AND 4) active
df_time = ctmc.event_list_to_timegrid(states)
custom_rule = lambda df: (
    ((df['node_1'] == 0) | (df['node_3'] == 0)) & 
    ((df['node_2'] == 0) | (df['node_4'] == 0))
)

inact, act = ctmc.calculate_collective_activity(df_time, condition_func=custom_rule)
ind_rating = 1 - (1-probs[0]*probs[2])*(1-probs[1]*probs[3])
print(f"{'% Active with real measurements:':<32} {act * 100:.2f}%")
print(f"{'% Active with assumptions:':<32} {ind_rating * 100:.2f}%")

# Anwsering More:

In [ ]:
from itertools import combinations
from scipy.stats import chi2

class RelationshipAnalysis:
    """
    Analysis of a two-state time series as DataFrames:
        node_summary : State exposure times, transition counts, marginal rates, and active fractions.
        pair_summary : Time-weighted pairwise association statistics.
        rate_dependence : Transition rates conditional on another node's state.
    """
    def __init__(self, df_time: pd.DataFrame, nodes:list, horizon:float=None, timestamp_col:str="timestamp"):
        self.df = df_time
        self.nodes = nodes
        self.horizon = horizon if horizon is not None else max(self.df[timestamp_col])
        self.timestamp_col = timestamp_col

    def analyze_ctmc_dependence(self):
        self.data = self._prepare_ctmc_data()
        node_summary = self._summarize_nodes()
        pair_summary = self._summarize_pairs()
        rate_dependence = self._summarize_conditional_rates()
        return node_summary, pair_summary, rate_dependence

    def _prepare_ctmc_data(self):
        """Validate the event timeline and construct state-duration intervals."""
        # Sort events and keep the final state recorded at duplicate timestamps.
        df = self.df
        df = (df.sort_values("timestamp").drop_duplicates("timestamp", keep="last").reset_index(drop=True))
        df["timestamp"] = df["timestamp"].astype(float)
        if df.empty or len(df) < 2:
            raise ValueError("At least two distinct timestamps are required.")
        times = df["timestamp"].to_numpy()
        states = df[self.nodes].to_numpy(dtype=int)
        if not np.isfinite(times).all():
            raise ValueError("Timestamps must be finite.")
        if not np.isin(states, [0, 1]).all():
            raise ValueError("All node states must be binary (0/1).")
        start = times[0]
        last_observed = times[-1]
        if self.horizon is None:
            self.horizon = last_observed
        else:
            self.horizon = float(self.horizon)
        if not np.isfinite(self.horizon) or self.horizon <= start:
            raise ValueError("self.horizon must be finite and exceed the first timestamp.")
        # Each row's state persists over [timestamp[k], timestamp[k+1]).
        left = times[:-1]
        right = np.minimum(times[1:], self.horizon)
        # Keep intervals overlapping the observation window.
        valid = (left < self.horizon) & (right > start)
        left = np.maximum(left[valid], start)
        right = right[valid]
        durations = right - left
        interval_states = states[:-1][valid]
        # Event k is the transition from states[k-1] to states[k].
        event_mask = (times[1:] > start) & (times[1:] <= self.horizon)
        return {
            "nodes": list(self.nodes),
            "times": times,
            "states": states,
            "interval_states": interval_states,
            "durations": durations,
            "event_times": times[1:][event_mask],
            "event_before": states[:-1][event_mask],
            "event_after": states[1:][event_mask],
            "start": start,
            "self.horizon": self.horizon,
        }

    def _summarize_nodes(self):
        """Calculate node exposure times, transition counts, and marginal rates."""
        rows = []
        data = self.data
        for i, node in enumerate(data["nodes"]):
            states = data["interval_states"][:, i]
            durations = data["durations"]
            # Total observed time spent in each state.
            time_in_0 = durations[states == 0].sum()
            time_in_1 = durations[states == 1].sum()
            # Count transitions using only events in the observation window.
            before = data["event_before"][:, i]
            after = data["event_after"][:, i]
            n_01 = int(np.sum((before == 0) & (after == 1)))
            n_10 = int(np.sum((before == 1) & (after == 0)))
            total_time = time_in_0 + time_in_1
            rows.append({
                "node": node,
                "time_in_0": time_in_0,
                "time_in_1": time_in_1,
                "n_0_to_1": n_01,
                "n_1_to_0": n_10,
                "q_0_to_1": (n_01 / time_in_0 if time_in_0 > 0 else np.nan),
                "q_1_to_0": (n_10 / time_in_1 if time_in_1 > 0 else np.nan),
                "active_fraction": (time_in_1 / total_time if total_time > 0 else np.nan),
            })
        return pd.DataFrame(rows)

    def _summarize_pairs(self):
        """Calculate time-weighted contemporaneous dependence for each node pair."""
        data = self.data
        rows = []
        durations = data["durations"]
        states = data["interval_states"]
        nodes = data["nodes"]
        for i, j in combinations(range(len(nodes)), 2):
            state_i = states[:, i]
            state_j = states[:, j]
            # Time spent in each joint state.
            both_active = durations[(state_i == 1) & (state_j == 1)].sum()
            i_active_j_inactive = durations[(state_i == 1) & (state_j == 0)].sum()
            i_inactive_j_active = durations[(state_i == 0) & (state_j == 1)].sum()
            total_time = durations.sum()
            p_i = ((both_active + i_active_j_inactive) / total_time if total_time > 0 else np.nan)
            p_j = ((both_active + i_inactive_j_active) / total_time if total_time > 0 else np.nan)
            p_j_given_i = (both_active / (both_active + i_active_j_inactive) if both_active + i_active_j_inactive > 0 else np.nan)
            rows.append({
                "node_i": nodes[i],
                "node_j": nodes[j],
                "time_both_active": both_active,
                "p_i_active": p_i,
                "p_j_active": p_j,
                "p_j_active_given_i_active": p_j_given_i,
                "risk_difference": (p_j_given_i - p_j if np.isfinite(p_j_given_i) and np.isfinite(p_j) else np.nan),
                "joint_active_excess": (both_active / total_time - p_i * p_j if total_time > 0 else np.nan),
            })
        return pd.DataFrame(rows)

    def _summarize_conditional_rates(self):
        """Estimate transition rates conditional on another node's state."""
        data = self.data
        rows = []
        nodes = data["nodes"]
        durations = data["durations"]
        interval_states = data["interval_states"]
        event_before = data["event_before"]
        event_after = data["event_after"]

        # Transition events classified by their pre-event state.
        for i, target in enumerate(nodes):
            for j, context in enumerate(nodes):
                if i == j:
                    continue
                target_state = interval_states[:, i]
                context_state = interval_states[:, j]
                for source in (0, 1):
                    destination = 1 - source
                    for context_value in (0, 1):
                        # Time at risk: target is in the source state
                        # while the context node has the specified state.
                        exposure_mask = ((target_state == source) & (context_state == context_value))
                        exposure = durations[exposure_mask].sum()
                        # Count transitions with matching pre-event states.
                        event_mask = ((event_before[:, i] == source) & (event_after[:, i] == destination) & (event_before[:, j] == context_value))
                        count = int(np.sum(event_mask))
                        rows.append({
                            "target": target,
                            "context": context,
                            "target_source_state": source,
                            "context_state": context_value,
                            "transition_count": count,
                            "exposure_time": exposure,
                            "transition_rate": (count / exposure if exposure > 0 else np.nan),
                            })
        return pd.DataFrame(rows)

    def plot_ctmc_node_summary(self, node_summary, nodes=None):
        """Visualize CTMC activity, transition-rate uncertainty, and data support."""
        if nodes is None:
            nodes = node_summary["node"].tolist()
        df = (node_summary.set_index("node") .reindex(nodes) .reset_index())
        labels = df["node"].tolist()
        x = np.arange(len(labels))
        # Prepare transition-rate estimates and confidence intervals
        rate_specs = [
            {"count": "n_0_to_1", "exposure": "time_in_0", "rate": "q_0_to_1", "label": "Activation (0→1)", "offset": -0.12,},
            {"count": "n_1_to_0", "exposure": "time_in_1", "rate": "q_1_to_0", "label": "Deactivation (1→0)", "offset": 0.12,},
        ]
        for spec in rate_specs:
            intervals = [
                self._poisson_rate_interval(int(count), exposure)
                if np.isfinite(count) and np.isfinite(exposure)
                else (np.nan, np.nan)
                for count, exposure in zip(df[spec["count"]], df[spec["exposure"]])
            ]
            df[spec["rate"] + "_lower"] = [v[0] for v in intervals]
            df[spec["rate"] + "_upper"] = [v[1] for v in intervals]

        # Figure layout
        fig, axes = plt.subplots(2, 2, figsize=(14, 9))
        ax_activity, ax_rates, ax_counts, ax_exposure = axes.flat

        # Empirical activity vs. fitted stationary probability
        activity = df["active_fraction"].to_numpy(dtype=float)
        ax_activity.bar(x, activity, alpha=0.75, label="Observed active fraction")
        q01 = df["q_0_to_1"].to_numpy(dtype=float)
        q10 = df["q_1_to_0"].to_numpy(dtype=float)
        stationary = np.divide(q01, q01 + q10, out=np.full_like(q01, np.nan), where=np.isfinite(q01 + q10) & ((q01 + q10) > 0),)
        ax_activity.scatter(x, stationary, marker="D", s=55, label="Fitted stationary probability", zorder=3,)
        ax_activity.set_xticks(x, labels, rotation=30)
        ax_activity.set_ylim(0, 1)
        ax_activity.set_ylabel("Probability / fraction")
        ax_activity.set_title("Observed vs. Fitted Activity")
        ax_activity.grid(axis="y", alpha=0.3)
        ax_activity.legend(fontsize=8)

        # Marginal transition rates and 95% intervals
        for spec in rate_specs:
            rate = df[spec["rate"]].to_numpy(dtype=float)
            lower = df[spec["rate"] + "_lower"].to_numpy(dtype=float)
            upper = df[spec["rate"] + "_upper"].to_numpy(dtype=float)
            valid = (np.isfinite(rate) & np.isfinite(lower) & np.isfinite(upper))
            positions = x[valid] + spec["offset"]
            ax_rates.errorbar(
                positions,
                rate[valid],
                yerr=np.vstack([rate[valid] - lower[valid], upper[valid] - rate[valid],]),
                fmt="o",
                capsize=4,
                label=spec["label"],
            )
        ax_rates.set_xticks(x, labels, rotation=30)
        ax_rates.set_ylabel("Rate per unit time")
        ax_rates.set_title("Transition Rates with 95% CIs")
        ax_rates.grid(axis="y", alpha=0.3)
        ax_rates.legend(fontsize=8)
        # Number of observed transitions
        width = 0.36
        ax_counts.bar(x - width / 2, df["n_0_to_1"], width, label="Activation",)
        ax_counts.bar(x + width / 2, df["n_1_to_0"], width, label="Deactivation",)
        ax_counts.set_xticks(x, labels, rotation=30)
        ax_counts.set_ylabel("Transition count")
        ax_counts.set_title("Evidence Supporting Rate Estimates")
        ax_counts.grid(axis="y", alpha=0.3)
        ax_counts.legend(fontsize=8)

        # Exposure time in each state
        ax_exposure.bar(x, df["time_in_0"], label="Inactive (state 0)",)
        ax_exposure.bar(x, df["time_in_1"], bottom=df["time_in_0"], label="Active (state 1)",)
        ax_exposure.set_xticks(x, labels, rotation=30)
        ax_exposure.set_ylabel("Time")
        ax_exposure.set_title("Observation Exposure by State")
        ax_exposure.grid(axis="y", alpha=0.3)
        ax_exposure.legend(fontsize=8)
        fig.suptitle("CTMC Node-Level Summary and Uncertainty", fontsize=14)
        fig.tight_layout()
        plt.show()
        
    def plot_ctmc_pairwise_dependence(self, pair_summary, nodes=None):
        """Plot pairwise risk differences and excess joint activity."""
        if nodes is None:
            nodes = list(dict.fromkeys(pair_summary["node_i"].tolist() + pair_summary["node_j"].tolist()))
        pair_metrics = [("risk_difference", "Change in P(j active) when i is active"), ("joint_active_excess", "Excess joint activity vs. independence"),]
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        for ax, (metric, title) in zip(axes, pair_metrics):
            if metric not in pair_summary.columns:
                ax.set_visible(False)
                continue
            matrix = pd.DataFrame(np.nan, index=nodes, columns=nodes, dtype=float)
            for _, row in pair_summary.iterrows():
                i, j = row["node_i"], row["node_j"]
                value = row[metric]
                if i in matrix.index and j in matrix.columns:
                    matrix.loc[i, j] = value
                    # Mirror only if this metric is symmetric.
                    matrix.loc[j, i] = value
            for node in nodes:
                matrix.loc[node, node] = 0.0
            values = matrix.to_numpy(dtype=float)
            finite = np.abs(values[np.isfinite(values)])
            limit = float(finite.max()) if finite.size else 1.0
            if limit == 0:
                limit = 1.0
            image = ax.imshow(values, cmap="RdBu_r", vmin=-limit, vmax=limit, aspect="equal",)
            ax.set_xticks(range(len(nodes)), nodes, rotation=45)
            ax.set_yticks(range(len(nodes)), nodes)
            ax.set_title(title)
            for i in range(len(nodes)):
                for j in range(len(nodes)):
                    value = values[i, j]
                    if np.isfinite(value):
                        ax.text(j, i, f"{value:.2f}", ha="center", va="center", fontsize=8,)
            fig.colorbar(image, ax=ax, shrink=0.8)
        fig.suptitle("Pairwise Dependence")
        fig.tight_layout()
        plt.show()

    def plot_ctmc_conditional_rates(self, rate_dependence, nodes=None):
        """Plot conditional activation and deactivation rate ratios"""
        if nodes is None:
            nodes = list(
                dict.fromkeys(rate_dependence["target"].tolist() + rate_dependence["context"].tolist()))
        rates = rate_dependence.copy()
        pivot = rates.pivot_table(index=["target", "context", "target_source_state"], columns="context_state", values="transition_rate", aggfunc="first",)
        if 0 not in pivot.columns or 1 not in pivot.columns:
            print("Conditional-rate plot skipped: both context states (0 and 1) are required.")
            return
        # Compute log2(rate when context=1 / rate when context=0) to avoid divide by 0
        rate_inactive = pivot[0]
        rate_active = pivot[1]
        valid = (rate_inactive.notna() & rate_active.notna() & (rate_inactive > 0) & (rate_active > 0))
        pivot["log2_rate_ratio"] = np.nan
        pivot.loc[valid, "log2_rate_ratio"] = np.log2(rate_active[valid] / rate_inactive[valid])
        for source_state, label in [(0, "Activation (0 → 1)"), (1, "Deactivation (1 → 0)"),]:
            try:
                subset = pivot.xs(source_state, level="target_source_state",)["log2_rate_ratio"].unstack("context")
            except KeyError:
                print(f"No data available for {label}.")
                continue
            subset = subset.reindex(index=nodes, columns=nodes)
            values = subset.to_numpy(dtype=float)
            finite = np.abs(values[np.isfinite(values)])
            limit = max(float(finite.max()), 1.0) if finite.size else 1.0
            fig, ax = plt.subplots(figsize=(max(7, 1.5 * len(nodes)), max(4, 1.0 * len(nodes)),))
            image = ax.imshow(values, cmap="RdBu_r", vmin=-limit, vmax=limit, aspect="auto",)
            ax.set_xticks(range(len(nodes)), nodes, rotation=45)
            ax.set_yticks(range(len(nodes)), nodes)
            ax.set_xlabel("Context node (0 = inactive, 1 = active)")
            ax.set_ylabel("Target node")
            ax.set_title(f"Conditional {label} Rate Ratio")
            for i in range(len(nodes)):
                for j in range(len(nodes)):
                    value = values[i, j]
                    if np.isfinite(value):
                        ax.text(j, i, f"{value:.2f}", ha="center", va="center", fontsize=8,)
            fig.colorbar(image, ax=ax, label="log₂(rate | context=1 / rate | context=0)",)
            fig.tight_layout()
            plt.show()

    @staticmethod
    def _poisson_rate_interval(count, exposure, confidence=0.95):
        """Garwood confidence interval for an event rate."""
        if exposure <= 0:
            return np.nan, np.nan
        alpha = 1 - confidence
        lower = (0.5 * chi2.ppf(alpha / 2, 2 * count) / exposure if count > 0 else 0.0)
        upper = (0.5 * chi2.ppf(1 - alpha / 2, 2 * (count + 1)) / exposure)
        return lower, upper


In [ ]:
nodes = ["node_1", "node_2", "node_3", "node_4"]
rel = RelationshipAnalysis(df_time, nodes)

In [ ]:
node_summary, pair_summary, rate_dependence = rel.analyze_ctmc_dependence()
display(node_summary.round(4),)
display(pair_summary.head().round(4))
display(rate_dependence.head().round(4))
rel.plot_ctmc_node_summary(node_summary)
rel.plot_ctmc_pairwise_dependence(pair_summary)
rel.plot_ctmc_conditional_rates(rate_dependence)

In [ ]:
df_time.shape

In [ ]:
class ProjectCTMC:
    """ Implimentation of a Bayesian Hierarchical CTMC Model """
    def __init__(self, df_time:pd.DataFrame, n_trajectories:int=500, block_duration:int=None, seed:int=123):
        self.df_time=df_time
        self.n_trajectories=n_trajectories
        self.block_duration=block_duration
        self.seed=seed
        self.nodes = [c for c in df_time.columns if c != 'timestamp']

    def bootstrap_ctmc_trajectories(self) -> list[pd.DataFrame]:
        rng = np.random.default_rng(self.seed)
        nodes = self.nodes
        df=self.df_time[["timestamp", *nodes]].sort_values("timestamp").drop_duplicates("timestamp", keep="last").reset_index(drop=True).copy()
        self._validate_df()
        times = df['timestamp'].to_numpy(dtype=float)
        states = df[nodes].to_numpy(dtype=np.int8)
        start, end = times[0], times[-1]
        horizon = end - start
        if self.block_duration is None:
            block_duration = horizon * 0.10
        else:
            block_duration = self.block_duration
        # Intervals represent the state held until the next event.
        interval_starts = times[:-1]
        interval_ends = times[1:]
        interval_states = states[:-1]
        trajectories = []
        for _ in range(self.n_trajectories):
            out_times = [start]
            out_states = [states[0].copy()]
            current_time = start
            while current_time < end:
                # Sample a block start from the observed time window.
                block_start = rng.uniform(start, end)
                block_end = min(block_start + block_duration, end)
                # Locate the observed state at the sampled block start.
                idx = np.searchsorted(interval_starts, block_start, side="right") - 1
                idx = np.clip(idx, 0, len(interval_states) - 1)
                # Map sampled block times into the synthetic time axis.
                block_times = [block_start]
                block_states = [interval_states[idx].copy()]
                k = idx + 1
                while k < len(interval_starts) and interval_starts[k] < block_end:
                    block_times.append(interval_starts[k])
                    block_states.append(interval_states[k].copy())
                    k += 1
                block_times.append(block_end)
                # Append the block, translating its time to the output timeline.
                for bt, bs in zip(block_times, block_states):
                    mapped_time = current_time + (bt - block_start)
                    if mapped_time <= end and mapped_time > out_times[-1]:
                        out_times.append(mapped_time)
                        out_states.append(bs.copy())
                current_time += block_end - block_start
            # Ensure the trajectory ends exactly at the observed horizon.
            if out_times[-1] < end:
                out_times.append(end)
                out_states.append(out_states[-1].copy())
            traj = pd.DataFrame(out_states, columns=nodes)
            traj.insert(0, "timestamp", out_times[:len(traj)])
            traj = traj[traj["timestamp"] <= end].reset_index(drop=True)
            # Remove duplicate times and retain a valid joint-state timeline.
            traj = (traj.sort_values("timestamp") .drop_duplicates("timestamp", keep="last") .reset_index(drop=True))
            trajectories.append(traj)
        return trajectories    

    def summarize_trajectory_distribution(self, trajectories):
        nodes = self.nodes
        rows = []
        for sim_id, traj in enumerate(trajectories):
            t = traj["timestamp"].to_numpy(dtype=float)
            if len(t) < 2 or t[-1] <= t[0]:
                continue
            # Each state persists until the next timestamp.
            durations = np.diff(t)
            states = traj[nodes].to_numpy()[:-1]
            for j, node in enumerate(nodes):
                active_time = durations[states[:, j] == 1].sum()
                total_time = durations.sum()
                rows.append({"simulation": sim_id, "node": node, "active_fraction": active_time / total_time,})
        results = pd.DataFrame(rows)
        return (results.groupby("node")["active_fraction"].agg(
            mean="mean", lower=lambda x: x.quantile(0.025), median="median", upper=lambda x: x.quantile(0.975), std="std",
            ) .reset_index())

    def plot_ctmc_trajectory(self, trajectory=None, ax=None, title="Node-state trajectory"):
        """Plot a binary network trajectory as a state raster."""
        nodes = self.nodes
        if trajectory is None:
            trajectory = self.df_time
        df = self.df_time[["timestamp", *nodes]].sort_values("timestamp")
        df = df.drop_duplicates("timestamp", keep="last")
        times = df["timestamp"].to_numpy(dtype=float)
        states = df[nodes].to_numpy(dtype=int)
        if len(times) < 2:
            raise ValueError("At least two timestamps are required.")
        # Append final time so each state occupies its full holding interval.
        edges = np.append(times, times[-1] + (times[-1] - times[-2]) * 0.001)
        if ax is None:
            _, ax = plt.subplots(figsize=(14, max(4, len(nodes) * 0.18)))
        ax.pcolormesh(edges, np.arange(len(nodes) + 1), states.T, cmap="coolwarm", vmin=0, vmax=1, shading="flat",)
        ax.set_yticks(np.arange(len(nodes)) + 0.5)
        ax.set_yticklabels(nodes)
        ax.set_xlabel("Time")
        ax.set_ylabel("Node")
        ax.set_title(title, weight='bold')
        return ax

    def plot_trajectory_comparison(self, trajectories, n_show=4):
        n_show = min(n_show, len(trajectories))
        _, axes = plt.subplots(n_show + 1, 1, figsize=(14, max(5, 2.5 * (n_show + 1))), sharex=True, constrained_layout=True,)
        self.plot_ctmc_trajectory(ax=axes[0], title="Observed trajectory",)
        for i in range(n_show):
            self.plot_ctmc_trajectory(trajectories[i], ax=axes[i + 1], title=f"Simulated trajectory {i + 1}",)
        plt.show()

    def plot_active_probability(self, trajectories, n_grid=200):
        """Plot estimated P(node active at time) across simulated trajectories."""
        nodes = self.nodes
        start = max(traj["timestamp"].min() for traj in trajectories)
        end = min(traj["timestamp"].max() for traj in trajectories)
        grid = np.linspace(start, end, n_grid)
        active = np.zeros((len(nodes), n_grid), dtype=float)
        for traj in trajectories:
            traj = traj.sort_values("timestamp").drop_duplicates("timestamp")
            t = traj["timestamp"].to_numpy(dtype=float)
            x = traj[nodes].to_numpy(dtype=int)
            idx = np.searchsorted(t, grid, side="right") - 1
            idx = np.clip(idx, 0, len(t) - 1)
            active += x[idx].T
        active /= len(trajectories)
        fig, ax = plt.subplots(figsize=(14, max(4, len(nodes) * 0.18)))
        time_edges = np.linspace(grid[0], grid[-1], len(grid) + 1)
        mesh = ax.pcolormesh(time_edges, np.arange(len(nodes) + 1), active, cmap="viridis", vmin=0, vmax=1, shading="flat",)
        ax.set_yticks(np.arange(len(nodes)) + 0.5)
        ax.set_yticklabels(nodes)
        ax.set_xlabel("Time")
        ax.set_ylabel("Node")
        ax.set_title("Estimated Node Active Probability")
        fig.colorbar(mesh, ax=ax, label="Probability active")
        plt.show()

    def _validate_df(self):
        times = self.df_time["timestamp"].to_numpy(dtype=float)
        states = self.df_time[self.nodes].to_numpy(dtype=np.int8)
        if len(times) < 3:
            raise ValueError("At least three distinct timestamps are required.")
        if not np.isfinite(times).all() or np.any(np.diff(times) <= 0):
            raise ValueError("Timestamps must be finite and strictly increasing.")
        if not np.isin(states, [0, 1]).all():
            raise ValueError("Node states must be binary (0/1).")



In [ ]:
proj = ProjectCTMC(df_time)
traj = proj.bootstrap_ctmc_trajectories()
active_frac = proj.summarize_trajectory_distribution(traj)
ax = proj.plot_ctmc_trajectory()
plt.show()

In [ ]:
proj.plot_trajectory_comparison(traj)

In [ ]:
proj.plot_active_probability(traj)

In [ ]:
from pathlib import Path
import nbformat

def ipynb_to_txt(ipynb_path, txt_path=None):
    """Export a Jupyter notebook to a reversible TXT representation."""
    ipynb_path = Path(ipynb_path)
    txt_path = Path(txt_path) if txt_path else ipynb_path.with_suffix(".txt")
    notebook = nbformat.read(ipynb_path, as_version=4)
    with txt_path.open("w", encoding="utf-8") as f:
        nbformat.write(notebook, f)
    return txt_path

def txt_to_ipynb(txt_path, ipynb_path=None):
    """Reconstruct a Jupyter notebook from its TXT representation."""
    txt_path = Path(txt_path)
    ipynb_path = (Path(ipynb_path) if ipynb_path else txt_path.with_suffix(".ipynb"))
    with txt_path.open("r", encoding="utf-8") as f:
        notebook = nbformat.read(f, as_version=4)
    nbformat.validate(notebook)
    nbformat.write(notebook, ipynb_path)
    return ipynb_path